# PlantSeg Round-2 — Notebook G (v2, instrumented): single-pass / flip-only / TTA breakdown, all 8 classifiers

**If you're re-running after a stall with zero log output:** this version prints a heartbeat at every single step (`flush=True` everywhere) and skips `pip install` if `timm` is already importable, so whatever happens next, the log will show exactly where it stopped instead of staying blank. Run cells one at a time (Shift+Enter), not "Run All" — if a cell doesn't print within ~30 seconds, that cell itself is the problem and you can stop immediately instead of waiting.

In [1]:
import sys
print("CELL 1 START", flush=True)
try:
    import timm
    print("timm already available, version", timm.__version__, flush=True)
except ImportError:
    print("timm not found, installing...", flush=True)
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "timm"], check=True)
    import timm
    print("timm installed, version", timm.__version__, flush=True)

import os, json, math, time, random, warnings, pickle, glob
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision
from PIL import Image
warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CELL 1 DONE -- Device:", device, flush=True)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0), flush=True)

CELL 1 START
timm already available, version 1.0.29
CELL 1 DONE -- Device: cuda
GPU: Tesla T4


In [2]:
print("CELL 2 START -- dataset + checkpoint discovery", flush=True)
DATA_ROOT = Path("/kaggle/input/datasets/alenmaruf/plantseg/plantseg")
if not (DATA_ROOT/"Metadata.csv").exists():
    for cand in [Path("/kaggle/input/plantseg"), Path("plantseg"), Path(".")]:
        if (cand/"Metadata.csv").exists(): DATA_ROOT = cand; break
print("DATA_ROOT resolved to:", DATA_ROOT, flush=True)
assert (DATA_ROOT/"Metadata.csv").exists(), f"Metadata.csv not found under {DATA_ROOT} -- attach the PlantSeg dataset and/or edit DATA_ROOT by hand, then re-run from here."
print("Metadata.csv found.", flush=True)

CKPT_FILENAMES = {
    "ConvNeXtV2Tiny":    "ConvNeXtV2Tiny_best.pt",
    "MobileNetV3Large":  "MobileNetV3Large_best.pt",
    "EfficientNetB0":    "EfficientNetB0_best.pt",
    "ResNet50":          "ResNet50_best.pt",
    "EfficientNetV2S":   "EfficientNetV2S_best.pt",
    "ViTBase16":         "ViTBase16_best.pt",
    "SwinTiny":          "SwinTiny_best.pt",
    "GatedFusionLight":  "GatedFusionLight_best.pt",
}

def find_ckpt(fname):
    hits = sorted(glob.glob(f"/kaggle/input/**/{fname}", recursive=True))
    if not hits:
        stem = fname.replace("_best.pt", "")
        hits = sorted(glob.glob(f"/kaggle/input/**/*{stem}*.pt", recursive=True))
    return hits[0] if hits else None

CKPT_PATHS = {}
print("Checkpoint discovery (this walks /kaggle/input once per filename -- if it hangs here, your attached dataset is unusually large/deep; tell Claude):", flush=True)
for name, fname in CKPT_FILENAMES.items():
    t0 = time.time()
    p = find_ckpt(fname)
    CKPT_PATHS[name] = p
    print(f"  {name:20s} -> {p if p else 'NOT FOUND -- will be skipped'}   ({time.time()-t0:.1f}s)", flush=True)

OUTDIR = Path("/kaggle/working/notebookG_results"); OUTDIR.mkdir(parents=True, exist_ok=True)
IMG_SIZE_CLS = 224
IMG_SIZE_CLS_TF = 224
IMG_SIZE_CONVNEXT = 384
MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
print("CELL 2 DONE", flush=True)

CELL 2 START -- dataset + checkpoint discovery
DATA_ROOT resolved to: /kaggle/input/datasets/alenmaruf/plantseg/plantseg
Metadata.csv found.
Checkpoint discovery (this walks /kaggle/input once per filename -- if it hangs here, your attached dataset is unusually large/deep; tell Claude):
  ConvNeXtV2Tiny       -> /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/CLassification/43seed_classifier_checkpoints/ConvNeXtV2Tiny_best.pt   (30.1s)
  MobileNetV3Large     -> /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/CLassification/43seed_classifier_checkpoints/MobileNetV3Large_best.pt   (11.1s)
  EfficientNetB0       -> /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/CLassification/43seed_classifier_checkpoints/EfficientNetB0_best.pt   (9.9s)
  ResNet50             -> /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/CLassification/43seed_classifier_checkpoints/ResNet50_best.pt   (0.1s)
  

In [3]:
print("CELL 3 START -- data index", flush=True)
meta = pd.read_csv(DATA_ROOT/"Metadata.csv", encoding="utf-8-sig")
meta["split"] = meta["Split"].map({"Training":"train","Validation":"val","Test":"test"})
meta["filepath"] = meta.apply(lambda r: DATA_ROOT/"images"/r["split"]/r["Name"], axis=1)
meta["label"] = meta["Disease"]
print(f"Metadata rows before filepath check: {len(meta)}", flush=True)
meta = meta[meta["filepath"].apply(lambda p: Path(p).exists())].reset_index(drop=True)
print(f"Metadata rows after filepath check: {len(meta)}", flush=True)
CLASS_NAMES = sorted(meta["label"].unique().tolist()); NUM_CLASSES = len(CLASS_NAMES)
label2idx = {c:i for i,c in enumerate(CLASS_NAMES)}
test_df = meta[meta.split=="test"].reset_index(drop=True)
print(f"classes={NUM_CLASSES} test={len(test_df)}", flush=True)
assert len(test_df) == 1561, f"Expected 1561 test images, got {len(test_df)} -- check DATA_ROOT/Metadata.csv, then re-run from here."

class DS(Dataset):
    def __init__(s, d, t): s.df = d.reset_index(drop=True); s.t = t
    def __len__(s): return len(s.df)
    def __getitem__(s, i):
        r = s.df.iloc[i]
        return s.t(Image.open(r.filepath).convert("RGB")), label2idx[r.label]
print("CELL 3 DONE", flush=True)

CELL 3 START -- data index
Metadata rows before filepath check: 7774
Metadata rows after filepath check: 7774
classes=115 test=1561
CELL 3 DONE


In [4]:
print("CELL 4 START -- model builders", flush=True)
def build_backbone(name, num_classes):
    if name == "MobileNetV3Large":
        m = torchvision.models.mobilenet_v3_large(weights=None)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes); return m, IMG_SIZE_CLS
    if name == "EfficientNetB0":
        return timm.create_model("efficientnet_b0", pretrained=False, num_classes=num_classes), IMG_SIZE_CLS
    if name == "ResNet50":
        m = torchvision.models.resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes); return m, IMG_SIZE_CLS
    if name == "EfficientNetV2S":
        return timm.create_model("tf_efficientnetv2_s", pretrained=False, num_classes=num_classes), IMG_SIZE_CLS
    if name == "ConvNeXtV2Tiny":
        return timm.create_model("convnextv2_tiny", pretrained=False, num_classes=num_classes), IMG_SIZE_CONVNEXT
    if name == "ViTBase16":
        return timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=num_classes), IMG_SIZE_CLS_TF
    if name == "SwinTiny":
        return timm.create_model("swin_tiny_patch4_window7_224", pretrained=False, num_classes=num_classes), IMG_SIZE_CLS_TF
    if name == "GatedFusionLight":
        class _Gated(nn.Module):
            def __init__(self, ba, da, bb, db, nc, gh=128, hh=256, dr=0.3):
                super().__init__(); self.branch_a=ba; self.branch_b=bb; fused=da+db
                self.gate = nn.Sequential(nn.Linear(fused,gh), nn.ReLU(), nn.Linear(gh,2), nn.Softmax(dim=1))
                self.classifier = nn.Sequential(nn.Linear(fused,hh), nn.ReLU(), nn.Dropout(dr), nn.Linear(hh,nc))
            def forward(self, x):
                fa, fb = self.branch_a(x), self.branch_b(x)
                g = self.gate(torch.cat([fa,fb], 1))
                return self.classifier(torch.cat([fa*g[:,0:1], fb*g[:,1:2]], 1))
        mb = torchvision.models.mobilenet_v3_large(weights=None)
        da = mb.classifier[0].in_features; mb.classifier = nn.Identity()
        b = timm.create_model("efficientnet_b0", pretrained=False, num_classes=0, global_pool="avg")
        return _Gated(mb, da, b, b.num_features, num_classes), IMG_SIZE_CLS_TF
    raise ValueError(name)

def safe_load_state(model, ckpt_path):
    obj = torch.load(ckpt_path, map_location="cpu")
    state = obj.get("state_dict", obj) if isinstance(obj, dict) and "state_dict" in obj else obj
    model_keys = set(model.state_dict().keys()); ckpt_keys = set(state.keys())
    missing = model_keys - ckpt_keys; unexpected = ckpt_keys - model_keys
    if len(missing) >= 0.5*len(model_keys):
        raise RuntimeError(f"{ckpt_path}: {len(missing)}/{len(model_keys)} keys missing -- wrong file or architecture.")
    model.load_state_dict(state, strict=False)
    return missing, unexpected
print("CELL 4 DONE", flush=True)

CELL 4 START -- model builders
CELL 4 DONE


In [5]:
print("CELL 5 START -- main eval loop", flush=True)
TABLE4_ACC = {
    "ConvNeXtV2Tiny": 0.784, "EfficientNetV2S": 0.761, "ViTBase16": 0.760, "SwinTiny": 0.752,
    "ResNet50": 0.744, "MobileNetV3Large": 0.734, "EfficientNetB0": 0.725, "GatedFusionLight": 0.691,
}
CHANCE = 1.0 / NUM_CLASSES
MISMATCH_TOLERANCE = 0.02

rows = []
for name, ckpt_path in CKPT_PATHS.items():
    t_model_start = time.time()
    print(f"\n--- {name}: starting ---", flush=True)
    if ckpt_path is None:
        print(f"[SKIP] {name}: no checkpoint found under /kaggle/input", flush=True); continue
    try:
        model, img_size = build_backbone(name, NUM_CLASSES)
        print(f"  {name}: architecture built, loading checkpoint from {ckpt_path} ...", flush=True)
        missing, unexpected = safe_load_state(model, ckpt_path)
        print(f"  {name}: checkpoint loaded (missing={len(missing)}, unexpected={len(unexpected)}), moving to {device} ...", flush=True)
        model = model.to(device).eval()
        print(f"  {name}: on {device}, starting inference over {len(test_df)} test images ...", flush=True)
    except Exception as e:
        print(f"[SKIP] {name}: failed to load -- {e}", flush=True); continue

    tf = transforms.Compose([transforms.Resize((img_size, img_size)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    loader = DataLoader(DS(test_df, tf), batch_size=32, shuffle=False, num_workers=0, pin_memory=(device.type=="cuda"))
    y_true=[]; pred_orig=[]; pred_flip=[]; pred_tta=[]
    n_batches = math.ceil(len(test_df)/32)
    with torch.no_grad():
        for bi, (x, y) in enumerate(loader):
            x = x.to(device)
            lg_o = model(x); lg_f = model(torch.flip(x, dims=[3]))
            pred_orig.extend(lg_o.argmax(1).cpu().numpy())
            pred_flip.extend(lg_f.argmax(1).cpu().numpy())
            pred_tta.extend((lg_o + lg_f).argmax(1).cpu().numpy())
            y_true.extend(y.numpy())
            if bi == 0 or (bi+1) % 10 == 0 or bi == n_batches-1:
                print(f"    {name}: batch {bi+1}/{n_batches}  ({time.time()-t_model_start:.1f}s elapsed)", flush=True)
    y_true=np.array(y_true); po=np.array(pred_orig); pf=np.array(pred_flip); pt=np.array(pred_tta)
    acc_o=(po==y_true).mean(); acc_f=(pf==y_true).mean(); acc_t=(pt==y_true).mean()
    agree=(po==pf).mean()
    corrected=int(((po!=y_true)&(pt==y_true)).sum()); harmed=int(((po==y_true)&(pt!=y_true)).sum())
    unchanged=int(len(y_true)-corrected-harmed)

    near_chance = acc_o < 2*CHANCE
    t4 = TABLE4_ACC.get(name); gap = (acc_t - t4) if t4 is not None else None
    flag = "NEAR-CHANCE -- checkpoint likely broken, exclude" if near_chance else \
           (f"MISMATCH vs Table 4 ({gap*100:+.1f} pt) -- investigate before using" if (gap is not None and abs(gap) > MISMATCH_TOLERANCE) else "OK, matches Table 4 within tolerance")

    rows.append({"model": name, "ckpt_path": ckpt_path, "n_test": len(y_true), "missing_keys": len(missing), "unexpected_keys": len(unexpected),
                 "acc_single_pass": float(acc_o), "acc_flip_only": float(acc_f), "acc_tta_averaged": float(acc_t),
                 "view_agreement_rate": float(agree), "n_corrected_by_tta": corrected, "n_harmed_by_tta": harmed,
                 "n_unchanged_by_tta": unchanged, "table4_reported_acc": t4, "gap_vs_table4": gap, "flag": flag})
    print(f"{name:20s} single={acc_o:.4f} flip={acc_f:.4f} tta={acc_t:.4f} agree={agree:.4f} "
          f"corrected={corrected} harmed={harmed} unchanged={unchanged}  [{flag}]  "
          f"(took {time.time()-t_model_start:.1f}s)", flush=True)
    del model
    if device.type == "cuda": torch.cuda.empty_cache()

df = pd.DataFrame(rows)
df.to_csv(OUTDIR/"tta_breakdown_all_classifiers_VERIFIED.csv", index=False)
print("\nCELL 5 DONE -- Saved:", OUTDIR/"tta_breakdown_all_classifiers_VERIFIED.csv", flush=True)

CELL 5 START -- main eval loop

--- ConvNeXtV2Tiny: starting ---
  ConvNeXtV2Tiny: architecture built, loading checkpoint from /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/CLassification/43seed_classifier_checkpoints/ConvNeXtV2Tiny_best.pt ...
  ConvNeXtV2Tiny: checkpoint loaded (missing=0, unexpected=0), moving to cuda ...
  ConvNeXtV2Tiny: on cuda, starting inference over 1561 test images ...
    ConvNeXtV2Tiny: batch 1/49  (6.1s elapsed)
    ConvNeXtV2Tiny: batch 10/49  (19.1s elapsed)
    ConvNeXtV2Tiny: batch 20/49  (33.9s elapsed)
    ConvNeXtV2Tiny: batch 30/49  (48.5s elapsed)
    ConvNeXtV2Tiny: batch 40/49  (62.5s elapsed)
    ConvNeXtV2Tiny: batch 49/49  (77.1s elapsed)
ConvNeXtV2Tiny       single=0.6765 flip=0.6803 tta=0.6855 agree=0.9129 corrected=28 harmed=14 unchanged=1519  [MISMATCH vs Table 4 (-9.9 pt) -- investigate before using]  (took 77.1s)

--- MobileNetV3Large: starting ---
  MobileNetV3Large: architecture built, loading checkpoint f

In [6]:
print("CELL 6 START -- final comparison", flush=True)
print(df[["model","acc_single_pass","acc_flip_only","acc_tta_averaged","view_agreement_rate",
          "n_corrected_by_tta","n_harmed_by_tta","n_unchanged_by_tta","table4_reported_acc","gap_vs_table4","flag"]].to_string(index=False))
print()
ok = df[df.flag.str.startswith("OK")]
mismatch = df[df.flag.str.contains("MISMATCH")]
broken = df[df.flag.str.contains("NEAR-CHANCE")]
print(f"\n{len(ok)} model(s) reproduce Table 4 within {MISMATCH_TOLERANCE*100:.0f} pt.")
print(f"{len(mismatch)} model(s) show a bigger gap than ConvNeXtV2-Tiny's already-disclosed one -- "
      f"DO NOT write these into the paper/response letter without first checking whether their "
      f"deposited checkpoint is the same vintage as the one behind Table 4's number (same "
      f"checkpoint-provenance issue, or a different problem).")
print(f"{len(broken)} model(s) near chance -- checkpoint did not train/load correctly, exclude from any TTA breakdown table.")
print("CELL 6 DONE", flush=True)

CELL 6 START -- final comparison
           model  acc_single_pass  acc_flip_only  acc_tta_averaged  view_agreement_rate  n_corrected_by_tta  n_harmed_by_tta  n_unchanged_by_tta  table4_reported_acc  gap_vs_table4                                                       flag
  ConvNeXtV2Tiny         0.676489       0.680333          0.685458             0.912876                  28               14                1519                0.784      -0.098542  MISMATCH vs Table 4 (-9.9 pt) -- investigate before using
MobileNetV3Large         0.681614       0.692505          0.689302             0.901986                  30               18                1513                0.734      -0.044698  MISMATCH vs Table 4 (-4.5 pt) -- investigate before using
  EfficientNetB0         0.673927       0.669443          0.668802             0.888533                  18               26                1517                0.725      -0.056198  MISMATCH vs Table 4 (-5.6 pt) -- investigate before using
       